Dataset Link: https://www.kaggle.com/datasets/ninadaithal/imagesoasis

This dataset requires lot of preprocessing because a single patients MRI Scan has 61 images. So we need to first separate all the distinct patient before we proceed

In [1]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import Input, Sequential, layers, mixed_precision  # type: ignore

mixed_precision.set_global_policy("mixed_float16")

In [3]:
print("Compute dtype: %s" % mixed_precision.global_policy().compute_dtype)
print("Variable dtype: %s" % mixed_precision.global_policy().variable_dtype)

Compute dtype: float16
Variable dtype: float32


In [4]:
data_dir = "./Data/MRI_Dataset"

filenames = []
labels = []

In [5]:
class_names = sorted(
    [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]
)
class_index = {class_names: idx for idx, class_names in enumerate(class_names)}

print(class_index)  # Class mapping

{'Mild Dementia': 0, 'Moderate Dementia': 1, 'Non Demented': 2, 'Very mild Dementia': 3}


In [6]:
for class_name in class_names:
    class_path = os.path.join(data_dir, class_name)
    current_label = class_index[class_name]
    for file in os.listdir(class_path):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            file_path = os.path.join(class_path, file)
            filenames.append(file_path)
            labels.append(current_label)


filenames = np.array(filenames)
labels = np.array(labels)
groups = np.array([os.path.basename(f)[:9] for f in filenames])

From the output of the code block below, we can see that class label 1: Moderate Dementia, we only have 488 images. One patient has multiple scans in this dataset so on an average each patient has 244 images. So Moderate Dementia only has data of 2 patients, which is useless for training.

In [7]:
values, counts = np.unique(labels, return_counts=True)
print(values, counts)

print(len(np.unique(groups)))

[0 1 2 3] [ 5002   488 67222 13725]
347


So, I am going to merge moderate dementia with mild dementia. Because we are removing moderate dementia it's label 1 is being changed to 0 and label 2 and label 3 needs to be renamed as label 1 and label 2 respectively

In [8]:
print("Original label distribution:", counts)

labels[labels == 1] = 0
labels[labels == 2] = 1
labels[labels == 3] = 2

new_values, new_counts = np.unique(labels, return_counts=True)
print("\n--- After Merging and Reindexing ---")
print(f"New unique labels: {new_values}")
print(f"New sample counts: {new_counts}")

Original label distribution: [ 5002   488 67222 13725]

--- After Merging and Reindexing ---
New unique labels: [0 1 2]
New sample counts: [ 5490 67222 13725]


Splitting the data into train, validation and test split

In [9]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=8)

train_val_index, test_index = next(gss.split(filenames, labels, groups=groups))
x, x_test = filenames[train_val_index], filenames[test_index]
y, y_test = labels[train_val_index], labels[test_index]
groups_train_val = groups[train_val_index]

train_index, val_index = next(gss.split(x, y, groups=groups_train_val))
x_train, x_val = x[train_index], x[val_index]
y_train, y_val = y[train_index], y[val_index]

To verify if GroupShuffleSplit actually split the data correctly

In [10]:
print(x_train[500], y_train[500])

./Data/MRI_Dataset/Mild Dementia/OAS1_0052_MR1_mpr-1_122.jpg 0


In [11]:
train_patients = set(groups_train_val[train_index])
val_patients = set(groups_train_val[val_index])
test_patients = set(groups[test_index])

intersection_trval = train_patients.intersection(val_patients)
intersection = set(groups[train_val_index]).intersection(test_patients)

print(f"Unique patients in Training Set: {len(train_patients)}")
print(f"Unique patients in Validation Set: {len(val_patients)}")
print(f"Unique patients in Testing Set: {len(test_patients)}")
print(
    f"Number of patients leaking into Train and Validation sets: {len(intersection_trval)}"
)
print(f"Number of patients leaking into Train_Val and Test: {len(intersection)}")

Unique patients in Training Set: 249
Unique patients in Validation Set: 45
Unique patients in Testing Set: 53
Number of patients leaking into Train and Validation sets: 0
Number of patients leaking into Train_Val and Test: 0


In [ ]:
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=1)
    img = tf.image.resize(img, [128, 128])
    img = img / 255.0

    label = tf.one_hot(label, depth=3)

    return img, label


train_dataset = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)
)  # Here x_train is the path of the image, y_train is the label4
train_dataset = train_dataset.map(load_and_preprocess)
train_dataset = train_dataset.shuffle(buffer_size=1000)
train_dataset = train_dataset.batch(64).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_dataset = (
    test_dataset.map(load_and_preprocess).batch(64).prefetch(tf.data.AUTOTUNE)
)


val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_dataset = val_dataset.map(load_and_preprocess).batch(64).prefetch(tf.data.AUTOTUNE)

echo "LD_LIBRARY_PATH=$(find $PWD/.venv -type d -name "lib" -path "*/nvidia/*" | paste -sd : -)" > .env

In [13]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices("GPU")))

Num GPUs Available:  1


Computing class weights to mitigate the effects of class imbalance

In [14]:
class_weight_array = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train), y=y_train
)

print(class_weight_array)

class_weight_dict = dict(enumerate(class_weight_array))
print(class_weight_dict)

[5.41935484 0.42477876 2.16774194]
{0: np.float64(5.419354838709677), 1: np.float64(0.4247787610619469), 2: np.float64(2.167741935483871)}


We have to be very careful during data augmentation here, because MRI Scans can be augmented is such way that would make it anatomically impossible. We keep padding=same because we want to look all the edges too. We use Flatten instead of Global Pooling layer because we want to preserve spatial relation 

In [20]:
from numpy import float32

model = Sequential()
model.add(Input(shape=(128, 128, 1)))
model.add(layers.RandomFlip("horizontal"))
model.add(layers.RandomRotation(factor=0.05))
model.add(layers.RandomZoom(height_factor=0.05, width_factor=0.05))

model.add(layers.Conv2D(32, (3, 3), activation="relu", padding="same"))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(pool_size=(2, 2)))
model.add(layers.Dropout(0.2))

model.add(layers.Conv2D(128, (3, 3), activation="relu", padding="same"))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(pool_size=(2, 2)))
model.add(layers.Dropout(0.2))

model.add(layers.Conv2D(256, (3, 3), activation="relu", padding="same"))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(pool_size=(2, 2)))
model.add(layers.Dropout(0.2))

model.add(layers.Flatten())

model.add(layers.Dense(128, activation="relu"))
model.add(layers.BatchNormalization())
model.add(layers.Dropout(0.5))

model.add(layers.Dense(3, activation="softmax", dtype=float32))
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip_2 (RandomFlip)      │ (None, 128, 128, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_2               │ (None, 128, 128, 1)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_2 (RandomZoom)      │ (None, 128, 128, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 128, 128, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 64, 64, 128)    │        36,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 64, 64, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 65536)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     8,388,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,723,779 (33.28 MB)

 Trainable params: 8,722,691 (33.27 MB)

 Non-trainable params: 1,088 (4.25 KB)

Cannot use categorical_crossentropy as labels are formatted as one dimensional np array. We need to use sparse_categorical_crossentropy. Drawback here is that we lose the soft label from categorical_crossentropy (Eg: 70% Dog, 20% Cat)

Edit: To use metrics like AUC, Precision and Recall, I need to use one hot encoding

Edit2: Apparently using one hot encoding with class weights absolutely wrecks the metrics, so back to using sparse_categorical_crossentropy

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    # metrics=[keras.metrics.AUC(), keras.metrics.Precision(), keras.metrics.Recall()],
    metrics=["sparse_categorical_accuracy"],
)

In [ ]:
history = model.fit(
    x=train_dataset,
    epochs=30,
    class_weight=class_weight_dict,
    validation_data=val_dataset,
)

Epoch 1/30
961/961 ━━━━━━━━━━━━━━━━━━━━ 44s 39ms/step - auc: 0.9916 - loss: 0.3321 - precision: 0.9559 - recall: 0.9416 - val_auc: 0.3594 - val_loss: 8.6078 - val_precision: 0.1458 - val_recall: 0.1458
Epoch 2/30
961/961 ━━━━━━━━━━━━━━━━━━━━ 35s 36ms/step - auc: 0.9852 - loss: 0.6855 - precision: 0.9457 - recall: 0.9221 - val_auc: 0.3594 - val_loss: 12.4350 - val_precision: 0.1458 - val_recall: 0.1458
Epoch 3/30
961/961 ━━━━━━━━━━━━━━━━━━━━ 35s 36ms/step - auc: 0.9868 - loss: 0.5655 - precision: 0.9342 - recall: 0.9132 - val_auc: 0.3594 - val_loss: 9.8024 - val_precision: 0.1458 - val_recall: 0.1458
Epoch 4/30
961/961 ━━━━━━━━━━━━━━━━━━━━ 35s 36ms/step - auc: 0.9847 - loss: 0.6450 - precision: 0.9376 - recall: 0.9078 - val_auc: 0.3594 - val_loss: 7.7055 - val_precision: 0.1458 - val_recall: 0.1458
Epoch 5/30
961/961 ━━━━━━━━━━━━━━━━━━━━ 35s 36ms/step - auc: 0.9758 - loss: 0.7009 - precision: 0.8945 - recall: 0.8854 - val_auc: 0.3594 - val_loss: 18.2362 - val_precision: 0.1458 - val_rec

In [ ]:
# Access the raw dictionary
metrics_dict = history.history

# Print the full list of your training and validation AUC across all epochs
print("Training AUC history:", metrics_dict["auc"])
print("Validation AUC history:", metrics_dict["val_auc"])

Training AUC history: [0.9915533065795898, 0.9851603507995605, 0.9867662787437439, 0.9847341179847717, 0.975796639919281, 0.9822403192520142, 0.9747406840324402, 0.9764120578765869, 0.9854671955108643, 0.9636919498443604, 0.8582413196563721, 0.7687098383903503, 0.83201664686203, 0.7230609059333801, 0.6643224358558655, 0.6565628051757812, 0.6694986820220947, 0.49874842166900635, 0.5824856758117676, 0.5228765606880188, 0.5171655416488647, 0.5322214961051941, 0.4661012589931488, 0.45824554562568665, 0.47016647458076477, 0.47687962651252747, 0.49758514761924744, 0.4915223717689514, 0.49081602692604065, 0.5059611797332764]
Validation AUC history: [0.359375, 0.359375, 0.359375, 0.359375, 0.359375, 0.359375, 0.359375, 0.359375, 0.359375, 0.359375, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125, 0.53125]
